# Step 7 — Model Selection & Why Random Forest
### Credit Card Underwriting Pipeline

---

## Why Model Selection Matters

> **There is no universally "best" model.** The best model depends on:
> - Size and type of your data (tabular vs image vs text)
> - Business requirements (interpretability vs pure accuracy)
> - Regulatory requirements (credit models must be explainable)
> - Training time budget
> - Whether the dataset is balanced or imbalanced

---

## Model Comparison for This Problem

| Model | Pros | Cons | Credit Use? |
|-------|------|------|-------------|
| **Logistic Regression** | Highly interpretable, fast | Can't capture non-linear patterns | Yes — traditional scorecard |
| **Decision Tree** | Interpretable, no scaling needed | Overfits badly | Rarely alone |
| **Random Forest** | Accurate, handles non-linearity, robust | Less interpretable (SHAP fixes this) | Yes — very common |
| **Gradient Boosting (XGBoost)** | Often highest accuracy | Slower, more tuning needed | Yes — common in fintech |
| **Neural Network** | Highest ceiling accuracy | Black box, needs lots of data | Rare in traditional banking |
| **SVM** | Works well in high dimensions | Slow on large data, hard to tune | Uncommon |

---

## Why We Choose Random Forest

**1. Handles tabular data well** — our data has 200 features of mixed types (numeric + categorical).
RF handles this naturally without special preprocessing.

**2. Non-linear patterns** — credit risk has complex interactions
(e.g., low income is only a problem if DTI is also high). RF captures these via tree splits.

**3. Built-in feature importance** — tells us which features drive decisions.
Essential for regulatory reporting.

**4. Robust to outliers and missing values** — trees split on rank, not exact value.

**5. SHAP-compatible** — Random Forests have an efficient SHAP implementation
(TreeExplainer) that works in seconds even on large forests.

**6. No feature scaling required** — trees split on thresholds; scale does not affect splits.
(We scale anyway to keep the pipeline consistent for potential future model comparisons.)

**7. Industry acceptance** — Random Forest is widely accepted in credit risk modelling,
with regulators comfortable with its explainability via feature importance + SHAP.

---

## The Bias-Variance Tradeoff

> **Bias** = model too simple → misses real patterns (underfitting)
> **Variance** = model too complex → memorises training data (overfitting)
>
> **Random Forest balances this by:**
> - Using many trees (reduces variance — errors average out)
> - Each tree trained on a random subset of data (bagging — reduces variance)
> - Each split considers a random subset of features (decorrelates trees)
> - Individual trees can be deep/complex (low bias)


In [ ]:
# ── Full pipeline re-run ─────────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd, time
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
SEED=42

DATA_PATH='cc_underwriting_5k_stratified11.csv'
IGNORE_COLS=['applicant_id','target_approved','target_credit_limit_assigned']
df_raw=pd.read_csv(DATA_PATH)
numeric_cols=[c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols=[c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]

def missing_report(df):
    m=df.isnull().sum();p=m/len(df)*100
    r=pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)

miss=missing_report(df_raw)
drop_cols=miss[miss['missing_pct']>50].index.tolist()
df=df_raw.drop(columns=drop_cols)
numeric_cols=[c for c in numeric_cols if c in df.columns]
cat_cols=[c for c in cat_cols if c in df.columns]
for col in numeric_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].median(),inplace=True)
for col in cat_cols:
    if df[col].isnull().sum()>0: df[col].fillna(df[col].mode()[0],inplace=True)
df['target']=(df['target_approved']=='Yes').astype(int)
df_fe=df.copy();eps=1e-6
df_fe['income_to_limit_ratio']=df_fe['annual_income']/(df_fe['requested_credit_limit']+eps)
df_fe['bureau_score_mean']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].mean(axis=1)
df_fe['bureau_score_std']=df_fe[['fico_score','equifax_score','experian_score','transunion_score']].std(axis=1)
df_fe['derogatory_x_dti']=df_fe['derogatory_marks_count']*df_fe['debt_to_income_ratio']
df_fe['monthly_net_income']=df_fe['monthly_income']-df_fe['total_monthly_expenses']
df_fe['total_risk_flags']=sum([(df_fe[f]=='Yes').astype(int) for f in ['prior_default_flag','prior_bankruptcy_flag','thin_file_flag'] if f in df_fe.columns])
for col in ['annual_income','net_worth','total_assets','requested_credit_limit']:
    if col in df_fe.columns: df_fe[col+'_log']=np.log1p(np.clip(df_fe[col],0,None))
le=LabelEncoder()
cat_for_model=[c for c in cat_cols if df_fe[c].nunique()<=30]
for col in cat_for_model: df_fe[col+'_enc']=le.fit_transform(df_fe[col].astype(str))
FINAL_FEATURES=[c for c in numeric_cols+
    ['income_to_limit_ratio','bureau_score_mean','bureau_score_std','derogatory_x_dti',
     'monthly_net_income','total_risk_flags']+
    [c+'_log' for c in ['annual_income','net_worth','total_assets','requested_credit_limit'] if c+'_log' in df_fe.columns]+
    [c+'_enc' for c in cat_for_model] if c in df_fe.columns]
X=df_fe[FINAL_FEATURES].fillna(0);y=df_fe['target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED,stratify=y)
X_train_sm,y_train_sm=SMOTE(random_state=SEED,k_neighbors=5).fit_resample(X_train,y_train)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train_sm)
X_test_sc=scaler.transform(X_test)
X_test_sc_df=pd.DataFrame(X_test_sc,columns=FINAL_FEATURES)
print(f'Ready: X_train={X_train_sc.shape}, X_test={X_test_sc.shape}')

## 7.1 — Model Comparison: Which Algorithm Wins?

> Before committing to Random Forest, we compare it against simpler baselines.
> This is good scientific practice — always verify that a complex model is worth the extra complexity.
>
> **We compare:**
> - Logistic Regression (the simplest binary classifier — our baseline)
> - Decision Tree (a single tree — no ensembling)
> - Random Forest (ensemble of trees — our chosen model)
>
> **Metric:** AUC-ROC (Area Under ROC Curve)
> - 0.5 = random guessing (no skill)
> - 1.0 = perfect classifier


In [ ]:
# Quick comparison: train 3 models and compare AUC
# NOTE: Using simple settings here for speed; Random Forest gets full tuning next
models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=SEED, C=1.0),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=6, random_state=SEED),
    'Random Forest'       : RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
comparison_results = []

print('Model Comparison (5-fold CV AUC):')
print()

for name, clf in models.items():
    t0 = time.time()
    # cross_val_score runs the full train/evaluate cycle K times
    scores = cross_val_score(clf, X_train_sc, y_train_sm,
                              scoring='roc_auc', cv=cv, n_jobs=-1)
    elapsed = time.time() - t0

    comparison_results.append({
        'Model': name,
        'CV AUC Mean': scores.mean(),
        'CV AUC Std' : scores.std(),
        'Train Time (s)': elapsed
    })
    print(f'  {name:<25} AUC: {scores.mean():.4f} +/- {scores.std():.4f}  ({elapsed:.1f}s)')

comp_df = pd.DataFrame(comparison_results).sort_values('CV AUC Mean', ascending=False)
print()
print('Winner:', comp_df.iloc[0]['Model'])

In [ ]:
# Visualise the comparison
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#bdc3c7', '#f39c12', '#2ecc71']
bars = ax.barh(comp_df['Model'], comp_df['CV AUC Mean'],
               xerr=comp_df['CV AUC Std'], color=colors,
               edgecolor='white', capsize=5)

ax.axvline(x=0.5, color='#e74c3c', linestyle='--', linewidth=1.5, label='Random baseline (AUC=0.5)')
ax.set_xlabel('Cross-Validation AUC', fontsize=12)
ax.set_title('Model Comparison: AUC-ROC\n(higher is better, error bars show std across 5 folds)',
             fontsize=13, fontweight='bold')
ax.set_xlim(0.4, 1.0)
ax.legend()

# Add value labels
for bar, val in zip(bars, comp_df['CV AUC Mean']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 7.2 — Baseline Random Forest

> **Always start with a baseline model** before tuning.
> The baseline gives you a reference point: tuning is only worth doing if it significantly
> beats the baseline on cross-validation.
>
> **Cross-validation (K-fold):**
> Split training data into K folds. Train on K-1 folds, evaluate on 1 fold. Repeat K times.
> Average the K scores. This gives a more reliable estimate than a single split.


In [ ]:
# Baseline Random Forest — sensible defaults, no tuning
rf_base = RandomForestClassifier(
    n_estimators=200,    # 200 trees (good default — diminishing returns above 500)
    max_depth=None,      # trees grow until all leaves are pure (may overfit)
    random_state=SEED,
    n_jobs=-1            # use all CPU cores
)

print('Training baseline Random Forest (200 trees, no depth limit)...')
t0 = time.time()
rf_base.fit(X_train_sc, y_train_sm)
print(f'Training time: {time.time()-t0:.1f}s')

# 5-fold cross-validation on training data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(rf_base, X_train_sc, y_train_sm,
                             scoring='roc_auc', cv=cv, n_jobs=-1)

print(f'Baseline CV AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Quick test set evaluation
y_base_prob = rf_base.predict_proba(X_test_sc)[:, 1]
base_test_auc = roc_auc_score(y_test, y_base_prob)
print(f'Baseline Test AUC: {base_test_auc:.4f}')

## 7.3 — Hyperparameter Tuning with RandomizedSearchCV

### What are Hyperparameters?

> **Model parameters** = values learned from data during training (e.g., the split thresholds in each tree)
> **Hyperparameters** = settings YOU choose before training that control how the model learns

Key Random Forest hyperparameters:

| Hyperparameter | Effect | Lower | Higher |
|---------------|--------|-------|--------|
| `n_estimators` | Number of trees | Faster but less stable | More accurate, slower |
| `max_depth` | Max tree depth | Simpler, less overfit | More complex, may overfit |
| `min_samples_split` | Min samples to split a node | Simpler trees | Finer splits |
| `min_samples_leaf` | Min samples in leaf node | Finer leaves | Smoother, more regularised |
| `max_features` | Features considered per split | Diverse trees (lower variance) | Individual trees stronger |

### Why RandomizedSearch instead of GridSearch?

> **GridSearch** tries every combination: 4 × 5 × 3 × 3 × 3 × 2 = 1,080 combinations × 5 folds = 5,400 fits.
> With `n_iter=30`, RandomizedSearch tries only 30 × 5 = 150 fits — **36x faster** with
> similar or better results (random sampling often finds good regions of the search space faster).


In [ ]:
# Define the hyperparameter search space
param_grid = {
    'n_estimators'     : [100, 200, 300, 500],       # number of trees
    'max_depth'        : [None, 5, 10, 15, 20],      # maximum tree depth
    'min_samples_split': [2, 5, 10],                  # min samples needed to split
    'min_samples_leaf' : [1, 2, 4],                   # min samples in leaf
    'max_features'     : ['sqrt', 'log2', 0.5],       # features per split
    'class_weight'     : ['balanced', None]           # handle imbalance
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=30,           # try 30 random combinations (increase for production)
    scoring='roc_auc',   # optimise for AUC (not accuracy — we have class imbalance)
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    random_state=SEED,
    verbose=1,
    n_jobs=-1
)

print('Running RandomizedSearchCV (30 iterations × 5 folds = 150 model fits)...')
t0 = time.time()
rf_search.fit(X_train_sc, y_train_sm)
print(f'Search complete in {time.time()-t0:.1f}s')
print()
print(f'Best CV AUC  : {rf_search.best_score_:.4f}')
print(f'Best params  :')
for k, v in rf_search.best_params_.items():
    print(f'  {k:<25}: {v}')

In [ ]:
# Use the best estimator found by RandomizedSearchCV
best_rf = rf_search.best_estimator_

# Final evaluation on held-out test set
y_pred       = best_rf.predict(X_test_sc)
y_pred_proba = best_rf.predict_proba(X_test_sc)[:, 1]

final_test_auc = roc_auc_score(y_test, y_pred_proba)
final_accuracy = accuracy_score(y_test, y_pred)

print('Final Tuned Random Forest:')
print(f'  Test AUC     : {final_test_auc:.4f}')
print(f'  Test Accuracy: {final_accuracy:.4f}')
print()
print(f'Improvement vs baseline:')
print(f'  AUC gain     : {final_test_auc - base_test_auc:+.4f}')
print()
print('The tuned model is now ready for confusion matrix and full evaluation.')

## Step 7 Complete — Model Selection Summary

**Why Random Forest won:**

| Criterion | Random Forest | Logistic Regression | Decision Tree |
|----------|--------------|--------------------|----|
| AUC on this data | Highest | Medium | Lower |
| Non-linear patterns | Yes | No | Yes (but overfits) |
| Feature importance | Yes | Coefficients | Yes |
| SHAP compatible | Yes | Yes | Yes |
| Regulatory acceptance | High | Very High | Medium |
| Tuning complexity | Medium | Low | Low |

**Hyperparameter Tuning:**
- Used RandomizedSearchCV with 30 iterations
- 5-fold stratified cross-validation
- Optimised for AUC (not accuracy) due to class imbalance

**Next:** `08_Confusion_Matrix.ipynb` — Deep dive into the confusion matrix and what each cell means
